# Export Lumis-1 Speaker Model to LiteRT (INT4)

This notebook exports the fine-tuned `lumis1-speaker-v1` (Granite 4.0 1B **Dense**) model to `.litertlm` (TensorFlow Lite) format with INT4 quantization.

**Target Engine:** LiteRT / MediaPipe LLM Inference (Custom Runtime)
**Quantization:** INT4 (Dynamic)
**Architecture:** Granite 4.0 Dense (Attention-only, wrapped in Hybrid config)

> **Note:** Granite uses a BPE tokenizer (`tokenizer.json`). Standard MediaPipe tasks expect SentencePiece (`.model`). We export the **Model Only**. A custom runtime wrapper is required to handle the BPE tokenizer.

In [ ]:
!pip install ai-edge-torch-nightly tensorflow-cpu transformers torch sentencepiece accelerate

In [ ]:
import torch
import ai_edge_torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import tensorflow as tf

print(f"PyTorch version: {torch.__version__}")
print(f"AI Edge Torch version: {ai_edge_torch.__version__}")

## 1. Load Model

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Resolve absolute path (Compatible with Windows/WSL/Linux)
MODEL_PATH = os.path.abspath("../models/speaker/lumis1-speaker-v1")
EXPORT_DIR = os.path.abspath("../models/speaker/lumis1-speaker-v1-litert")
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f"Loading model from: {MODEL_PATH}")

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True
)
model.eval()

# Load Tokenizer (for verification only)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("Model and Tokenizer loaded successfully.")

## 2. Define Quantization Config (INT4)

In [ ]:
from ai_edge_torch.quantize import quant_recipes

# Select Quantization Recipe
# Try for INT4 specific recipe if available, else fall back to dynamic INT8 or custom config.
# Common recipes: full_int8_dynamic_quantization, full_int8_training_static_quantization

# Note: 'generative_int4_dynamic' is often the target for LLMs.
try:
    quant_config = quant_recipes.generative_int4_quantization()
    print("Using Generative INT4 Quantization recipe.")
except AttributeError:
    print("INT4 recipe not found directly. Using 'full_int8_dynamic_quantization' as fallback backbone, custom config may be needed.")
    # Fallback or Custom Definitions would go here.
    quant_config = quant_recipes.full_int8_dynamic_quantization()
    # Note: For strict INT4, specific low-level config is required if recipe missing.

# Optional: Sample input for tracing
sample_text = "Hello, how are you?"
sample_input_ids = tokenizer(sample_text, return_tensors="pt").input_ids
sample_args = (sample_input_ids,)


## 3. Convert and Export

In [ ]:
OUTPUT_FILE = os.path.join(EXPORT_DIR, "lumis1-speaker-v1-int4.tflite")

print("Starting Conversion... (This may take several minutes)")

# Convert
# ai_edge_torch.convert traces the model and applies quantization.
edge_model = ai_edge_torch.convert(
    model,
    sample_args,
    quant_config=quant_config
)

# Save
edge_model.export(OUTPUT_FILE)
print(f"Export complete! Saved to: {OUTPUT_FILE}")

## 4. Tokenizer Handling
Granite uses `tokenizer.json` (BPE). Standard LiteRT uses SentencePiece. Since we cannot convert automatically:
1. We copy `tokenizer.json` to the export folder.
2. The deployment runtime must load `tokenizer.json` using `tokenizers` library (Rust/C++/Python) instead of the MediaPipe built-in SentencePiece processor.

In [ ]:
import shutil

TOKENIZER_SRC = os.path.join(MODEL_PATH, "tokenizer.json")
TOKENIZER_DST = os.path.join(EXPORT_DIR, "tokenizer.json")
CONFIG_SRC = os.path.join(MODEL_PATH, "tokenizer_config.json")
CONFIG_DST = os.path.join(EXPORT_DIR, "tokenizer_config.json")

if os.path.exists(TOKENIZER_SRC):
    shutil.copy(TOKENIZER_SRC, TOKENIZER_DST)
    print(f"Copied tokenizer.json to {EXPORT_DIR}")
else:
    print("WARNING: tokenizer.json not found!")

if os.path.exists(CONFIG_SRC):
    shutil.copy(CONFIG_SRC, CONFIG_DST)
    print(f"Copied tokenizer_config.json to {EXPORT_DIR}")